# "가격을 맞혀봐요!" 캡스톤 프로젝트

이번 주 목표 - Amazon 데이터 스크랩을 기반으로 상품 설명에서 가격을 예측하는 모델 만들기

# 진행 순서

1일차: 데이터 수집 및 정제  
2일차: 데이터 전처리  
3일차: 평가, 기준 모델, 전통적 ML  
4일차: 딥러닝과 LLM  
5일차: 프론티어 모델 파인튜닝  

## 3일차: 평가, 기준 모델, 전통적 ML

오늘은 상품 가격을 예측하는 간단한 모델들을 만들어 봅니다

모델 성능을 평가하는 방법을 사용합니다

그리고 전통적인 머신러닝을 활용한 기준 모델들을 테스트합니다

In [ ]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

In [ ]:
LITE_MODE = False

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
def random_pricer(item):
    return random.randrange(1,1000)

In [ ]:
random.seed(42)
evaluate(random_pricer, test)

In [ ]:
# 재미있었나요?
# 더 잘할 수 있습니다 - 또 다른 단순한 모델을 만들어 봅시다

training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

In [ ]:
evaluate(constant_pricer, test)

In [ ]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [ ]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [ ]:
# 전통적 선형 회귀!

np.random.seed(42)

# 특성(features)과 타겟(target) 분리
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# 선형 회귀 모델 학습
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"절편(Intercept): {model.intercept_}")

# 테스트 세트 예측 및 평가
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"평균 제곱 오차(MSE): {mse}")
print(f"결정계수(R²): {r2}")

In [ ]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [ ]:
evaluate(linear_regression_pricer, test)

In [ ]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)


In [ ]:
# 불용어(stop words)를 제외한 가장 많이 등장하는 단어 1,000개:

selected_words = vectorizer.get_feature_names_out()
print(f"선택된 단어 수: {len(selected_words)}")
print("선택된 단어들:", selected_words[1000:1020])

In [ ]:
regressor = LinearRegression()
regressor.fit(X, prices)

In [ ]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [ ]:
evaluate(natural_language_linear_regression_pricer, test)

In [ ]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

## 랜덤 포레스트 모델

랜덤 포레스트는 **앙상블(ensemble)** 알고리즘의 일종으로, 여러 개의 작은 알고리즘을 결합하여 더 나은 예측을 만들어냅니다.

내부적으로는 **결정 트리(decision tree)** 라는 간단한 머신러닝 알고리즘을 사용합니다. 결정 트리는 입력의 특성 값을 검사하여 예측을 수행합니다. IF 구문이 있는 순서도와 비슷합니다. 결정 트리는 매우 빠르고 단순하지만 과적합(overfitting)되는 경향이 있습니다.

우리의 경우 "특성(features)"은 벡터의 요소들, 즉 상품 설명에서 특정 단어가 몇 번 등장하는지를 나타냅니다.

예를 들면 이렇습니다:

**결정 트리**  
\- "TV"라는 단어가 3회 이상 등장하면  
-- "LED"라는 단어가 2회 이상 등장하면  
--- "HD"라는 단어가 1회 이상 등장하면  
---- 가격 = $500


랜덤 포레스트는 여러 개의 결정 트리를 생성합니다. 각 트리는 데이터의 다른 무작위 부분집합과 다른 무작위 특성 부분집합으로 학습됩니다. 위에서 100개의 트리를 지정했는데, 이것이 기본값입니다.

그런 다음 랜덤 포레스트 모델은 모든 트리의 평균을 내어 최종 결과를 산출합니다.

In [ ]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [ ]:
evaluate(random_forest, test)

In [ ]:
# 모델을 저장하려는 경우 (특히 더 큰 데이터셋으로 실행 시 유용)

# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

## XGBoost 소개

랜덤 포레스트와 마찬가지로, XGBoost도 여러 결정 트리를 결합한 앙상블 모델입니다.

하지만 랜덤 포레스트와 달리, XGBoost는 트리를 하나씩 순차적으로 만들며 '경사 하강법(gradient descent)'을 사용하여 이전 트리의 오류를 다음 트리가 보정합니다.

랜덤 포레스트보다 훨씬 빠르므로 전체 데이터셋에서도 실행할 수 있으며, 일반화 성능도 보통 더 뛰어납니다.

**이 임포트가 작동하지 않으면 건너뛰세요! 필수 사항이 아닙니다. Mac에서는 터미널에서 `brew install libomp`를 실행해야 할 수 있습니다.**

In [ ]:
import xgboost as xgb

In [ ]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

In [ ]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [ ]:
evaluate(xg_boost, test)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">비즈니스 활용</h2>
            <span style="color:#181;">전통적 ML은 과거 학습용으로만 유용한 것이 아니라, 오늘날에도 산업 현장에서 특히 명확하게 식별 가능한 특성이 있는 작업에 널리 활용됩니다. 알고리즘을 탐색하고 실험하는 데 시간을 투자할 가치가 있습니다. 전통적 ML로 제 수치를 이길 수 있는지 도전해보세요! 저는 전체 800,000개 학습 데이터셋으로 랜덤 포레스트를 실행했습니다. 약 15시간이 걸렸고, 최종 오차는 $56.40까지 낮아졌습니다. 전통적 ML도 훌륭한 성능을 발휘할 수 있습니다 - 직접 도전해보세요.</span>
        </td>
    </tr>
</table>